In [35]:
git init

SyntaxError: invalid syntax (2830201818.py, line 1)

# Project 2

Author: Cameron Mullaney

This is the second project for ST554.

DESCRIPTION AND NARRATIVE

# Part I

Part I is split up into two major tasks :
    A. The creation of my "Proj2Script.py" file
    B. The use of the SparkDataCheck class and the methods created within. 
    
Piece A is outlined thoroughly in the script file, and piece B is below.
I will be shwocasing the methods created in the script to analyze and validate data from a csv of air quality data.

Loading in the tools I'll need, as well as my "Proj2Script.py" file from piece A

In [1]:
import pandas as pd
import numpy as np
import math
from pyspark.sql import SparkSession
from Proj2Script import *

Setting up my spark session, and reading in the data! 

In [3]:
spark = SparkSession.builder.appName('my_app').getOrCreate()

In [4]:
airdata = SparkDataCheck.fromcsv("air.csv", spark)
airdata.df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-23 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-23 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-23 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-23 21:00:00|   2.2|       137

26/03/23 20:54:05 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-cmullan@ncsu.edu/air.csv


### Initial Data Cleaning

There seem to be a couple issues here: 
1. I'm noticing this data has some -200 values in it, which correspond to missing values - Let's replace those with NULL values. \
I've designed the Proj2Script to properly handle NULL values, so this should ensure our methods work properly.
2. Python isn't happy with the header format. Let's rename some columns for clarity. We can get rid of the first column which is serving as a secondary index.
This is code pulled from our first project, and updated to match the current data processing method.
3. Our "Time" column seems to have assigned today's date to the times. We'll remove that.

In [5]:
airdata.df =  airdata.df.withColumnRenamed("C6H6(GT)", "B")\
                        .withColumnRenamed("PT08.S1(CO)", "CO")\
                        .withColumnRenamed("PT08.S2(NMHC)", "NMHC")\
                        .withColumnRenamed("PT08.S3(NOx)", "NOx")\
                        .withColumnRenamed("PT08.S4(NO2)", "NO2")\
                        .withColumnRenamed("PT08.S5(O3)", "O3")

airdata.df = airdata.df.drop("_c0")

Here, we're replacing those -200 values with NULL.

In [6]:
airdata.df = airdata.df.replace(float(-200), None) ## Replace "-200" with "NaN"

In [7]:
airdata.df.show()

+---------+-------------------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|               Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+-------------------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|2026-03-23 18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-23 19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|2026-03-23 20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-23 21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|2026-03-23 22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|2026-03-23 23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|

In [8]:
airdata.df.show()

+---------+-------------------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|               Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+-------------------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|2026-03-23 18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|2026-03-23 19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|2026-03-23 20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|2026-03-23 21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|2026-03-23 22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|2026-03-23 23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|

Great! Looks like our values have been replaced! To be sure, let's run minmax.

In [9]:
airdata.minmax()

<class 'NoneType'>
Column Provided: False
Grouping Var Provided: False


,0
min(CO(GT)),0.1000
max(CO(GT)),11.9000
min(CO),647.0000
max(CO),2040.0000
min(NMHC(GT)),7.0000
max(NMHC(GT)),1189.0000
min(B),0.1000
max(B),63.7000
min(NMHC),383.0000
max(NMHC),2214.0000


Perfect! No -200 values to be seen!

Now, for the Time column:

In [10]:
airdata.df = airdata.df.withColumn("Time", F.date_format(F.col("Time"), "HH:mm:ss"))

In [11]:
airdata.df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|3/11/2004|00:00:00|   1.2|1185|      31| 3.6| 690|     62|1462|     77|1333| 733|11.3|56.8|0.7603|


Great! Now we have a date, and a time column. \
Finally, I am going to make a list of our column names for use later.

In [12]:
OGcols = airdata.df.columns
OGcols

['Date',
 'Time',
 'CO(GT)',
 'CO',
 'NMHC(GT)',
 'B',
 'NMHC',
 'NOx(GT)',
 'NOx',
 'NO2(GT)',
 'NO2',
 'O3',
 'T',
 'RH',
 'AH']

## Method Testing

Here, I give some examples of the methods I've written, to show that they work.
I will provide 4-5 examples for each.

### Validation Methods

#### withinlimits()

This method takes in an item of SparkDataCheck class, a specific column name from that item's df attribute, and an upper and lower numeric bound. \
Then, a new column is appended providing True or False values for each row in the specified column, depending on if it's in the specified range. \
Only one bound is required, and if only one is provided a one-sided check will be made.
This method returns the edited SparkDataCheck item, so .show() must be used to see the table.

##### Numeric column specified, upper and lower bound supplied:

Here, you can see a new column on the right, corresponding to if each value of T is within the specified 10-11 range. \
This column is added to the SparkDataCheck item in use.

In [13]:
airdata.withinlimits("T", lower = 10, upper = 11).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|T within (10,11)|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|           false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|           false|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|           false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|            true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|           false|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|

##### Numeric column specified, only lower bound supplied:

Here, you can see a new column on the right, corresponding to if each value of T is below the given value 1.5. \
This column is added to the SparkDataCheck item in use.

In [14]:
airdata.withinlimits("CO(GT)", lower = 1.5).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|T within (10,11)|CO(GT) above 1.5|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|           false|            true|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|           false|            true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|           false|            true|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|            true|            true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|120

##### Non-numeric column specified, lower bound supplied:

When a non-numeric column is specified, the method makes no changes and returns your object unedited. A message is printed, telling you what's wrong.

In [15]:
airdata.withinlimits("T within (10,11)", lower = 50).df.show()

Non-numeric column supplied - No actions taken
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|T within (10,11)|CO(GT) above 1.5|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|           false|            true|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|           false|            true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|           false|            true|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|            true|            true|
|3/10/2004|22:0

##### Numeric column specified, only upper bound given:

This is similar to the earlier example with CO(GT). With only an upper bound given, every value <= 1200 is True, while all others are False.

In [16]:
airdata.withinlimits("CO", upper = 1200).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+-------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|T within (10,11)|CO(GT) above 1.5|CO below 1200|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+----------------+----------------+-------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|           false|            true|        false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|           false|            true|        false|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|           false|            true|        false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|            t

Just for ease of viewing, I will remove these additional columns now.

In [17]:
airdata.df = airdata.df.select(OGcols)
airdata.df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|3/11/2004|00:00:00|   1.2|1185|      31| 3.6| 690|     62|1462|     77|1333| 733|11.3|56.8|0.7603|


#### onlist()

This method takes in an item of SparkDataCheck class, a specific column name from that item's df attribute, and a list of strings. \
Then, a new column is appended providing True or False values for each row in the specified column, depending on if that value is on the list. \
This method returns the edited SparkDataCheck item, so .show() must be used to see the table.

##### String column supplied, string supplied:

Here, you can see a new column on the right, corresponding to if each cell in "Date" is 3/10/2004. \
This column is added to the SparkDataCheck item in use.

In [18]:
airdata.onlist("Date", ["3/10/2004"]).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|Date on list|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|        true|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|        true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|        true|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|        true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|        true|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.78

##### String column supplied, list of strings supplied:

Here, you can see a new column on the right, corresponding to if each cell in "Time" is in the list we provided.

In [19]:
airdata.onlist("Time", ["19:00:00", "21:00:00", "09:00:00"]).df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|Date on list|Time on list|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|        true|       false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|        true|        true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|        true|       false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|        true|        true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|        true|      

##### String column specified, non-numeric list supplied:

In this case, the method will determine that the list contains non-strings, print the issue, and return the unedited object.

In [20]:
airdata.onlist("Time", [18,19,20]).df.show()

Non-string list supplied - No actions taken
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|Date on list|Time on list|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|        true|       false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|        true|        true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|        true|       false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|        true|        true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|14

##### Non-string column specified, valid list given:

In this case, it will return the unedited object and print the issue.

In [21]:
airdata.onlist("B", ["11.9", "4.7"]).df.show()

Non-string column supplied - No actions taken
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|Date on list|Time on list|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+------------+------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|        true|       false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|        true|        true|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|        true|       false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|        true|        true|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|

##### Once again, clearing the new columns

In [22]:
airdata.df = airdata.df.select(OGcols)
airdata.df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|1393| 949|11.2|59.2|0.7848|
|3/11/2004|00:00:00|   1.2|1185|      31| 3.6| 690|     62|1462|     77|1333| 733|11.3|56.8|0.7603|


#### nulltest()

This method takes in an item of SparkDataCheck class and a specific column name from that item's df attribute. \
Then, a new column is appended providing True or False values for each row in the specified column, depending on if that value is NULL. \
This method returns the edited SparkDataCheck item, so .show() must be used to see the table.

In [23]:
airdata.nulltest("NO2(GT)").df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+---------------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|NO2(GT) is null|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+---------------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|          false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|          false|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|          false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172|1092|    122|1584|1203|11.0|60.0|0.7867|          false|
|3/10/2004|22:00:00|   1.6|1272|      51| 6.5| 836|    131|1205|    116|1490|1110|11.2|59.6|0.7888|          false|
|3/10/2004|23:00:00|   1.2|1197|      38| 4.7| 750|     89|1337|     96|

This method can be used pretty quickly to check counts of null values using .groupBy().

In [24]:
airdata.nulltest("NOx(GT)").df.groupBy("Time", "NOx(GT) is null").count().sort("Time").show()

+--------+---------------+-----+
|    Time|NOx(GT) is null|count|
+--------+---------------+-----+
|00:00:00|          false|  336|
|00:00:00|           true|   54|
|01:00:00|           true|   56|
|01:00:00|          false|  334|
|02:00:00|           true|   58|
|02:00:00|          false|  332|
|03:00:00|           true|  364|
|03:00:00|          false|   26|
|04:00:00|           true|   58|
|04:00:00|          false|  332|
|05:00:00|           true|   57|
|05:00:00|          false|  333|
|06:00:00|           true|   56|
|06:00:00|          false|  334|
|07:00:00|           true|   56|
|07:00:00|          false|  334|
|08:00:00|           true|   61|
|08:00:00|          false|  329|
|09:00:00|          false|  334|
|09:00:00|           true|   56|
+--------+---------------+-----+
only showing top 20 rows


In [25]:
airdata.nulltest("CO(GT)").df.groupBy("CO(GT) is null").count().show()

+--------------+-----+
|CO(GT) is null|count|
+--------------+-----+
|          true| 1683|
|         false| 7674|
+--------------+-----+



In [26]:
airdata.nulltest("B").df.show()

+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+---------------+---------------+--------------+---------+
|     Date|    Time|CO(GT)|  CO|NMHC(GT)|   B|NMHC|NOx(GT)| NOx|NO2(GT)| NO2|  O3|   T|  RH|    AH|NO2(GT) is null|NOx(GT) is null|CO(GT) is null|B is null|
+---------+--------+------+----+--------+----+----+-------+----+-------+----+----+----+----+------+---------------+---------------+--------------+---------+
|3/10/2004|18:00:00|   2.6|1360|     150|11.9|1046|    166|1056|    113|1692|1268|13.6|48.9|0.7578|          false|          false|         false|    false|
|3/10/2004|19:00:00|   2.0|1292|     112| 9.4| 955|    103|1174|     92|1559| 972|13.3|47.7|0.7255|          false|          false|         false|    false|
|3/10/2004|20:00:00|   2.2|1402|      88| 9.0| 939|    131|1140|    114|1555|1074|11.9|54.0|0.7502|          false|          false|         false|    false|
|3/10/2004|21:00:00|   2.2|1376|      80| 9.2| 948|    172

### Summarization Methods

The methods below take in a SparkDataCheck object, and analyze it in different ways. Each returns a pandas data frame.

#### minmax()

This method accepts a specific column name, as well as a grouping variable, `groupvar`. It will return a pandas df containing the minimum and maximum values for the specified column within the SparkDataCheck df attribute, grouped by `groupvar` if one is provided. \
If no column is provided, this method will return a df of the min and max values for every numeric column in the object's df attribute, grouped if specified.

In [27]:
airdata.minmax("T")

<class 'str'>
Column Provided: True
Grouping Var Provided: False


,min(T),max(T)
0,-1.9,44.6


As I did in the "Data Cleaning" section above, you can also use `.minmax()` to check for missing values in the data. Because -200 is far below the possible values for these variables, it would appear as the min value for any column with a missing value - using `.minmax()` and seeing no -200 values showed me that our replace method worked.

In [29]:
airdata.minmax()

<class 'NoneType'>
Column Provided: False
Grouping Var Provided: False


,0
min(CO(GT)),0.1000
max(CO(GT)),11.9000
min(CO),647.0000
max(CO),2040.0000
min(NMHC(GT)),7.0000
max(NMHC(GT)),1189.0000
min(B),0.1000
max(B),63.7000
min(NMHC),383.0000
max(NMHC),2214.0000


Here, I am chaining our `minmax()` method with a `.sort_values()` call, which helps sort the data by the column we want.

In [32]:
airdata.minmax("NOx", "Time").sort_values("Time")

<class 'str'>
Column Provided: True
Grouping Var Provided: True


,Time,min(NOx),max(NOx)
8,00:00:00,443,1627
1,01:00:00,435,1900
0,02:00:00,497,1955
13,03:00:00,530,2081
14,04:00:00,508,2331
2,05:00:00,515,2559
4,06:00:00,534,2294
7,07:00:00,407,2683
15,08:00:00,370,2327
5,09:00:00,345,2318


This method can be useful when combined with the validation methods from earlier: \
I can create and use a new category in one line - Here, finding the min and max levels for RH at different B levels.

In [ ]:
airdata.withinlimits("B", upper = 5).minmax("RH", groupvar = "B below 5")

<class 'str'>
Column Provided: True
Grouping Var Provided: True


,B below 5,min(RH),max(RH)
0,None,NaN,NaN
1,True,10.0,87.2
2,False,9.2,88.7


#### strcount()